# Feedback Storage Comparison

This Colab notebook compares feedback storage strategies for IDS outputs: JSON logs, SQLite, and vector memory storage.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

## 3) Load Dataset

In [ ]:
import os
import shutil
import json
import sqlite3
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG_SEED = 42
np.random.seed(RNG_SEED)

data_path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print('Dataset files:', os.listdir(data_path))

X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy')).reshape(-1)
if X_test.ndim == 3 and X_test.shape[-1] == 1:
    X_test = X_test[..., 0]

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

## 4) Simulate Attack Detection Results

In [ ]:
def simulate_feedback_records(X, y, n_samples=500, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(y), size=min(n_samples, len(y)), replace=False)

    attack_map = {0: 'BENIGN', 1: 'DOS', 2: 'PORTSCAN', 3: 'BRUTEFORCE', 4: 'DDOS'}
    records = []
    for sample_id, i in enumerate(idx):
        label = int(y[i])
        attack_type = attack_map.get(label, f'ATTACK_{label}')

        row = X[i]
        base = float(np.clip(np.mean(np.abs(row)), 0.0, 1.0))
        risk_score = float(np.clip(0.6 * base + 0.3 * (label > 0) + rng.normal(0, 0.08), 0.0, 1.0))

        if risk_score > 0.8:
            decision = 'BLOCK_IP'
        elif risk_score > 0.6:
            decision = 'RATE_LIMIT'
        elif risk_score > 0.4:
            decision = 'ALERT_ADMIN'
        else:
            decision = 'ALLOW'

        if attack_type == 'BENIGN':
            human_feedback = 'false_alarm' if decision != 'ALLOW' else 'confirmed_benign'
        else:
            human_feedback = 'confirmed_attack'

        records.append({
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'sample_id': int(sample_id),
            'attack_type': attack_type,
            'risk_score': float(risk_score),
            'decision': decision,
            'human_feedback': human_feedback,
            'features': row[:32].astype(float).tolist(),
        })
    return records

records = simulate_feedback_records(X_test, y_test, n_samples=500, seed=RNG_SEED)
print('Simulated records:', len(records))
records[0]

## Storage Backends: JSON, SQLite, Vector Memory

In [ ]:
class JSONStorage:
    def __init__(self, file_path):
        self.file_path = file_path
        os.makedirs(os.path.dirname(file_path), exist_ok=True)

    def store(self, record):
        with open(self.file_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=True) + '\n')

    def retrieve_recent(self, limit=100):
        if not os.path.exists(self.file_path):
            return []
        with open(self.file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()[-limit:]
        return [json.loads(line) for line in lines]


class SQLiteStorage:
    def __init__(self, db_path):
        self.db_path = db_path
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS feedback (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT,
                sample_id INTEGER,
                attack_type TEXT,
                risk_score REAL,
                decision TEXT,
                human_feedback TEXT
            )
            '''
        )
        self.conn.commit()

    def store(self, record):
        self.conn.execute(
            '''
            INSERT INTO feedback (timestamp, sample_id, attack_type, risk_score, decision, human_feedback)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                record['timestamp'],
                record['sample_id'],
                record['attack_type'],
                record['risk_score'],
                record['decision'],
                record['human_feedback'],
            ),
        )
        self.conn.commit()

    def retrieve_recent(self, limit=100):
        cur = self.conn.cursor()
        cur.execute(
            '''
            SELECT timestamp, sample_id, attack_type, risk_score, decision, human_feedback
            FROM feedback
            ORDER BY id DESC
            LIMIT ?
            ''',
            (limit,),
        )
        rows = cur.fetchall()
        return [
            {
                'timestamp': r[0],
                'sample_id': r[1],
                'attack_type': r[2],
                'risk_score': r[3],
                'decision': r[4],
                'human_feedback': r[5],
            }
            for r in rows
        ]


class VectorMemoryStorage:
    def __init__(self, prefix):
        self.vector_path = prefix + '_vectors.npy'
        self.meta_path = prefix + '_meta.jsonl'
        os.makedirs(os.path.dirname(self.vector_path), exist_ok=True)
        self.vectors = []
        self.metadata = []

    @staticmethod
    def _embed(features):
        arr = np.asarray(features, dtype=np.float32)
        if arr.size == 0:
            arr = np.zeros(8, dtype=np.float32)
        return np.array([
            np.mean(arr), np.std(arr), np.max(arr), np.min(arr),
            np.median(arr), np.percentile(arr, 75), np.percentile(arr, 25), np.linalg.norm(arr)
        ], dtype=np.float32)

    def store(self, record):
        self.vectors.append(self._embed(record.get('features', [])))
        self.metadata.append({
            'timestamp': record['timestamp'],
            'sample_id': record['sample_id'],
            'attack_type': record['attack_type'],
            'risk_score': record['risk_score'],
            'decision': record['decision'],
            'human_feedback': record['human_feedback'],
        })

    def finalize(self):
        if self.vectors:
            np.save(self.vector_path, np.vstack(self.vectors))
        with open(self.meta_path, 'w', encoding='utf-8') as f:
            for m in self.metadata:
                f.write(json.dumps(m, ensure_ascii=True) + '\n')

    def retrieve_recent(self, limit=100):
        return self.metadata[-limit:]

## Benchmark: Storage Time, Retrieval Time, Query Latency

In [ ]:
work_dir = '/content/feedback_storage_benchmark'
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)
os.makedirs(work_dir, exist_ok=True)

results = []

# JSON
json_store = JSONStorage(os.path.join(work_dir, 'feedback.jsonl'))
t0 = time.perf_counter()
for r in records:
    json_store.store(r)
storage_time = time.perf_counter() - t0

t0 = time.perf_counter()
_ = json_store.retrieve_recent(limit=200)
retrieval_time = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    _ = json_store.retrieve_recent(limit=50)
query_latency = (time.perf_counter() - t0) / 200

results.append({
    'Method': 'JSON Log Storage',
    'StorageTimeSec': storage_time,
    'RetrievalTimeSec': retrieval_time,
    'QueryLatencySec': query_latency,
})

# SQLite
sqlite_store = SQLiteStorage(os.path.join(work_dir, 'feedback.db'))
t0 = time.perf_counter()
for r in records:
    sqlite_store.store(r)
storage_time = time.perf_counter() - t0

t0 = time.perf_counter()
_ = sqlite_store.retrieve_recent(limit=200)
retrieval_time = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    _ = sqlite_store.retrieve_recent(limit=50)
query_latency = (time.perf_counter() - t0) / 200

results.append({
    'Method': 'SQLite Database Storage',
    'StorageTimeSec': storage_time,
    'RetrievalTimeSec': retrieval_time,
    'QueryLatencySec': query_latency,
})

# Vector Memory
vector_store = VectorMemoryStorage(os.path.join(work_dir, 'feedback_vector'))
t0 = time.perf_counter()
for r in records:
    vector_store.store(r)
vector_store.finalize()
storage_time = time.perf_counter() - t0

t0 = time.perf_counter()
_ = vector_store.retrieve_recent(limit=200)
retrieval_time = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    _ = vector_store.retrieve_recent(limit=50)
query_latency = (time.perf_counter() - t0) / 200

results.append({
    'Method': 'Vector Memory Storage',
    'StorageTimeSec': storage_time,
    'RetrievalTimeSec': retrieval_time,
    'QueryLatencySec': query_latency,
})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='QueryLatencySec', ascending=True).reset_index(drop=True)
results_df

## Comparison Table

In [ ]:
results_df.style.format({
    'StorageTimeSec': '{:.6f}',
    'RetrievalTimeSec': '{:.6f}',
    'QueryLatencySec': '{:.6f}'
})

## Charts

In [ ]:
sns.set(style='whitegrid')

plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x='Method', y='StorageTimeSec', palette='Blues_d')
plt.title('Storage Time Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x='Method', y='RetrievalTimeSec', palette='Greens_d')
plt.title('Retrieval Time Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x='Method', y='QueryLatencySec', palette='Reds_d')
plt.title('Query Latency Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
best = results_df.iloc[0]
print(f"Best method by query latency: {best['Method']}")
print(best.to_string())